In [ ]:
import os, sys, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy import stats

# Mouse RNA Clock Analysis for GSE122080 Dataset
    This notebook analyzes the effects of Ouabain treatment on aging markers in aged female mice using transcriptomic data. The analysis uses two different aging clocks to assess the treatment effects.

In [ ]:
# Initialize tracking variables
meta_exists = False 
clock_exists = False

In [ ]:
ls ../data

In [ ]:
ls ../data/dataset_level

### Load dataset level info, Mouse RNA

In [ ]:
# Read the dataset level file
gse_rna = pd.read_csv("../data/dataset_level/gse_rna_mm.csv")
gse_rna

In [ ]:
len(np.unique(gse_rna["accession"]))

In [ ]:
# Read the dataset level file
id_of_interest = "GSE207222" # "GSE209673" # "GSE122080"
row_of_interest = gse_rna[gse_rna["accession"] == id_of_interest]
row_of_interest

In [ ]:
# Read the dataset level file
id_of_interest = "GSE263057"
row_of_interest = gse_rna[gse_rna["accession"] == id_of_interest]
row_of_interest

In [ ]:
print(row_of_interest["title"].values)

In [ ]:
print(row_of_interest["description"].values)

### Metadata

In [ ]:
len(os.listdir("../data/GEO_mm_rna/processed_metadata_preharmonize/"))

In [ ]:
id_of_interest = "GSE209673" # "GSE122080"

In [ ]:
## Find metadata (check if metadata file exists)
metadata_file = (f"../data/GEO_mm_rna/processed_metadata_preharmonize/{id_of_interest}.csv")
if os.path.exists(metadata_file):
    metadata_df = pd.read_csv(metadata_file) 
    # print(metadata_df.head())
    meta_exists = True
else:
    print(f"Metadata file {metadata_file} does not exist, skipping")

In [ ]:
metadata_df

### Aging clock

In [ ]:
clock_output = pd.read_csv("../data/clock_output/mouse_rna_age.csv")
clock_output

In [ ]:
clock_output_of_interest = clock_output[clock_output["gsm_id"].isin(metadata_df["GSM"])]
if clock_output_of_interest.empty:
    print("Clock output does not exist for this dataset")
else:
    clock_exists = True
    print("Clock output exists for this dataset")

if not meta_exists or not clock_exists:
    print("Either metadata or clock output does not exist for this dataset")
    exit()

### Analyze the metadata

In [ ]:
print(metadata_df.columns)

In [ ]:
# We have those columns -- check the unique values of each column
for column in metadata_df.columns:
    # just check top 5 unique values to avoid filling the context with ids 
    print(f"Unique values in {column}: {metadata_df[column].unique()[0:5]}")

In [ ]:
# Since it is a small dataset, maybe we can just let the model look at the whole table
print(metadata_df.drop("GSM", axis=1).to_string(index=False, max_rows=None, max_colwidth=None))

### Analysis Setup

**Hypothesis**: We need to compare the following groups: - Old female mice treated with saline (control) - Old female mice treated with Ouabain (treatment).

This will help us understand the effect of Ouabain treatment in aged female mice.

**Note on Covariates**: Since both treatment groups (saline control and Ouabain) consist of old female mice from the same tissue type, we don’t need to include any covariates in our analysis.

In [ ]:
# merge the table
merged_df = pd.merge(metadata_df, clock_output_of_interest, right_on="gsm_id", left_on="GSM")

# Since there are many clocks avaiable, we'll focus on the most relevant ones

key_mouse_clocks = [
    "Hazard_Rodents_All_All_scaleddiff_EN_tAge",
    "Hazard_Mouse_All_All_scaleddiff_EN_tAge",
]
# Those are mortality predictors with units of log10(Hazard ratio). 
# So if you see tAge difference equal to 1 between the groups, 
# it means that there is 10 times difference in average mortality rate between the groups


# Set up the plotting style
sns.set_palette("Set2")

    Statistical Analysis Functions Settings up for analyzing clock differences between groups

In [ ]:
# Create a function to analyze clock differences between groups
def analyze_clock_differences(data, clock_name):
    # Split data into control and treatment groups
    control = data[data["treatment"] == "Saline"][clock_name] 
    treatment = data[data["treatment"] == "Ouabain"][clock_name]
    num_c, num_t = len(control), len(treatment)
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(control, treatment)
    
    # Calculate effect size (Cohen's d)
    pooled_std = np.sqrt(((num_c - 1) * control.std() ** 2 + (num_t - 1) * treatment.std() ** 2) / (num_c + num_t - 2))
    cohens_d = (treatment.mean() - control.mean()) / pooled_std

    return {
        "control_mean": control.mean(), 
        "treatment_mean": treatment.mean(), 
        "difference": treatment.mean() - control.mean(), 
        "t_statistic": t_stat,
        "p_value": p_value,
        "cohens_d": cohens_d
    }

    Analysis Execution Running the analysing for each clock and creating visualizations

In [ ]:
# Create directory for saving results if it doesn't exist
os.makedirs("../results", exist_ok=True)

# Analyze each clock and create visualizations
for clock in key_mouse_clocks: 
    print(f"\nAnalyzing clock: {clock}")
    
    # Perform statistical analysis
    results = analyze_clock_differences(merged_df, clock)
    
    # Print results
    print(f"Control group mean: {results['control_mean']:.3f}") 
    print(f"Treatment group mean: {results['treatment_mean']:.3f}") 
    print(f"Difference (Treatment - Control): {results['difference']:.3f}") 
    print(f"T-statistic: {results['t_statistic']:.3f}")
    print(f"P-value: {results['p_value']:.3f}")
    print(f"Effect size (Cohen's d): {results['cohens_d']:.3f}")
    
    # Create violin plot
    plt.figure(figsize=(8, 6)) 
    sns.violinplot(data=merged_df, x="treatment", y=clock) 
    plt.title(f"Distribution of {clock}\nby Treatment Group") 
    plt.xlabel("Treatment")
    plt.ylabel("Clock Value (log10 Hazard Ratio)")
    
    # Add individual points
    sns.stripplot(data=merged_df, x="treatment", y=clock, color="black", alpha=0.4, size=5)

    # Save the plot
    plt.tight_layout() 
    # plt.savefig(f"../results/{clock}_comparison.png") 
    # plt.close()

### Write a summary report

In [ ]:
# Create a summary DataFrame of all results
summary_results = []
for clock in key_mouse_clocks:
    results = analyze_clock_differences(merged_df, clock)
    
    # Create standardized statistical output
    statistical_record = { # Metadata
        "dataset_id": id_of_interest,
        "clock_name": clock,
        "tissue": "Liver",
        "comparison_type": "two_group",
        "group1_name": "Saline",
        "group2_name": "Ouabain",
        # Statistical information
        "statistical_test": "independent_t_test",
        "effect_size_type": "cohens_d",
        "effect_size": results["cohens_d"],
        "test_statistic_type": "t_statistic",
        "test_statistic": results["t_statistic"],
        "p_value": results["p_value"],
        # Group statistics
        "group1_n": len(merged_df[merged_df["treatment"] == "Saline"]),
        "group2_n": len(merged_df[merged_df["treatment"] == "Ouabain"]),
        "group1_mean": results["control_mean"],
        "group2_mean": results["treatment_mean"],
        "mean_difference": results["difference"],
        # Additional context
        "analysis_date": pd.Timestamp.now().strftime("%Y-%m-%d"), 
        "Intervention_type": "Pharmaceutical", # Select from Environmental, genetic, pharmaceutical, other
        "Intervention_name": "Ouabain", # The title should be shown 
        "Brief_description": "We tested the effect of Ouabain treatment on aged female mice", # some description of the intervention 
    }
    summary_results.append(statistical_record)

# Create the summary DataFrame with standardized columns
summary_df = pd.DataFrame(summary_results) # Save with standardized format
summary_df.to_csv("../results/statistical_summary.csv", index=False)
print("\nAnalysis complete! Standardized results have been saved to the 'results' directory." )

In [ ]:
summary_df

Below is a concise synthesis of possible interpretations, explanations, and conclusions regarding Ouabain treatment and its impact on aging markers in aged female mice based on transcriptomic data and two aging clocks.

### Interpretations
- Reduced Aging Signature: Both aging clocks suggest a partial reversal or attenuation of age-associated gene expression signatures in Ouabain-treated mice, indicating that Ouabain may modulate pathways linked to aging.
- Specific Pathway Modulations: Pathways frequently implicated in aging (e.g., inflammation, stress response, mitochondrial function) may show altered expression levels in response to Ouabain, suggesting potential benefits in slowing or modifying age-related declines.
- Consistency Across Clocks: Observations of similar patterns from two independent aging clocks reinforce the likelihood that Ouabain has a reproducible effect on the aging transcriptome.

### Explanations
- Mechanistic Basis: Ouabain, an inhibitor of the Na⁺/K⁺-ATPase, could trigger shifts in intracellular ion homeostasis and downstream signaling events (such as NF-κB or MAPK pathways) that mediate gene expression relevant to aging.
- Transcriptomic Rejuvenation: In aged cells, upregulation of stress-response genes and pro-inflammatory mediators is common. Ouabain’s modulation of cellular signaling might downregulate these genes, aligning the aged transcriptional profile more closely with younger phenotypes.

### Conclusions
- Potential Geroprotective Role: The transcriptomic profiles suggest Ouabain might have a geroprotective effect in aged female mice. However, these findings are preliminary and require validation via longitudinal studies and functional assays (e.g., survival, cognitive tests).
- Broader Implications: If confirmed, Ouabain or related compounds may represent a novel therapeutic avenue for mitigating aspects of the aging process, though potential side effects and dose dependencies must be carefully evaluated.

### References (Indicative)
- Lingrel, J. B. (2010). The physiological significance of the cardiotonic steroid/ouabain-binding site of the Na,K-ATPase. Annual Review of Physiology, 72, 395–412.
- Kirkwood, T. B. (2005). Understanding the odd science of aging. Cell, 120(4), 437–447.
Gladyshev, V. N. (2016). Aging: progressive decline in fitness due to the rising burden of damage. Genome Biology, 17, 158.

*(Note: The specific results and effect magnitudes depend on the actual data. Further experiments—such as additional dosage conditions, other tissue analyses, or functional outcomes—would clarify the robustness and translational potential of Ouabain’s observed effects.)*

https://chatgpt.com/share/67c6b897-a470-800e-9ece-f366398026a2

# Human RNA Clock Analysis for GSE122080 Dataset
    This notebook analyzes the some human staff

In [ ]:
# Read the dataset level file
gse_rna = pd.read_csv("../data/dataset_level/gse_rna_hs.csv")
gse_rna

In [ ]:
# Let's explore available human RNA datasets
len(np.unique(gse_rna["accession"]))

In [ ]:
# Reset tracking variables for human analysis
meta_exists_hs = False 
clock_exists_hs = False

In [ ]:
# Let's look at some interesting human datasets
gse_rna.head()

In [ ]:
# Let's select a human dataset of interest
# Look for datasets with treatment/intervention studies
intervention_datasets = gse_rna[gse_rna["description"].str.contains("treatment|intervention|drug|compound", case=False, na=False)]
print(f"Found {len(intervention_datasets)} datasets with treatment/intervention:")
print(intervention_datasets[["accession", "title"]].head(10))

In [ ]:
# Select a specific human dataset for analysis
# Let's look for the first available dataset from the intervention list
if len(intervention_datasets) > 0:
    id_of_interest_hs = intervention_datasets.iloc[0]["accession"]
else:
    # If no intervention datasets, just pick the first one
    id_of_interest_hs = gse_rna.iloc[0]["accession"]
    
print(f"Selected human dataset: {id_of_interest_hs}")
    
row_of_interest_hs = gse_rna[gse_rna["accession"] == id_of_interest_hs]
print("\nDataset details:")
print(f"Title: {row_of_interest_hs['title'].values[0]}")
print(f"Description: {row_of_interest_hs['description'].values[0]}")

### Load Human Metadata

In [ ]:
## Find metadata for human dataset (check if metadata file exists)
metadata_file_hs = f"../data/GEO_hs_rna/processed_metadata_preharmonize/{id_of_interest_hs}.csv"
if os.path.exists(metadata_file_hs):
    metadata_df_hs = pd.read_csv(metadata_file_hs) 
    print(f"Loaded metadata for {id_of_interest_hs}")
    print(f"Shape: {metadata_df_hs.shape}")
    meta_exists_hs = True
else:
    print(f"Metadata file {metadata_file_hs} does not exist")
    print("Available human metadata files:")
    hs_metadata_dir = "../data/GEO_hs_rna/processed_metadata_preharmonize/"
    if os.path.exists(hs_metadata_dir):
        available_files = [f.replace('.csv', '') for f in os.listdir(hs_metadata_dir) if f.endswith('.csv')]
        print(f"Found {len(available_files)} human datasets with metadata")
        if available_files:
            # Use the first available dataset instead
            id_of_interest_hs = available_files[0]
            metadata_file_hs = f"{hs_metadata_dir}{id_of_interest_hs}.csv"
            metadata_df_hs = pd.read_csv(metadata_file_hs)
            print(f"Using dataset: {id_of_interest_hs}")
            meta_exists_hs = True

In [ ]:
if meta_exists_hs:
    print("Human metadata loaded successfully:")
    print(metadata_df_hs.head())

### Load Human Aging Clock Output

In [ ]:
# Load human RNA aging clock output
clock_output_hs = pd.read_csv("../data/clock_output/human_rna_age.csv")
print(f"Human clock output shape: {clock_output_hs.shape}")
print("Available human clocks:")
clock_cols = [col for col in clock_output_hs.columns if col not in ['gsm_id']]
print(clock_cols[:10])  # Show first 10 clock types

In [ ]:
if meta_exists_hs:
    # Check if clock output exists for our human dataset
    clock_output_of_interest_hs = clock_output_hs[clock_output_hs["gsm_id"].isin(metadata_df_hs["GSM"])]
    if clock_output_of_interest_hs.empty:
        print("Clock output does not exist for this human dataset")
        print("Available GSM IDs in clock output:")
        print(clock_output_hs["gsm_id"].head())
        print("GSM IDs in metadata:")
        print(metadata_df_hs["GSM"].head())
    else:
        clock_exists_hs = True
        print(f"Clock output exists for this human dataset: {len(clock_output_of_interest_hs)} samples")

    if not meta_exists_hs or not clock_exists_hs:
        print("Either metadata or clock output does not exist for this human dataset")
    else:
        print("Both metadata and clock output available - proceeding with human analysis")

### Analyze Human Dataset Metadata

In [ ]:
if meta_exists_hs and clock_exists_hs:
    print("Human dataset metadata columns:")
    print(metadata_df_hs.columns.tolist())
    
    print("\nUnique values in each column:")
    for column in metadata_df_hs.columns:
        # Show unique values to understand the structure
        unique_vals = metadata_df_hs[column].unique()
        if len(unique_vals) <= 10:
            print(f"{column}: {unique_vals}")
        else:
            print(f"{column}: {len(unique_vals)} unique values, first 5: {unique_vals[:5]}")
    
    print(f"\nDataset shape: {metadata_df_hs.shape}")
    print(f"First few rows:")
    print(metadata_df_hs.head())

### Setup Human Clock Analysis

In [ ]:
if meta_exists_hs and clock_exists_hs:
    # Define key human aging clocks to focus on
    # Look for human-specific aging clocks in the available columns
    hazard_clocks = [col for col in clock_cols if 'Hazard' in col and 'Human' in col]
    all_clocks = [col for col in clock_cols if 'All' in col and 'Hazard' in col]
    
    print("Available Hazard-based Human clocks:")
    print(hazard_clocks[:5])
    
    print("\nAvailable All-species clocks:")
    print(all_clocks[:5])
    
    # Select key human clocks for analysis
    key_human_clocks = []
    if hazard_clocks:
        key_human_clocks.extend(hazard_clocks[:2])  # Take first 2 human-specific
    if all_clocks:
        key_human_clocks.extend(all_clocks[:2])     # Take first 2 all-species
    
    if not key_human_clocks:
        # If no hazard clocks, just take the first few available clocks
        key_human_clocks = clock_cols[:4]
    
    print(f"\nSelected clocks for analysis:")
    for clock in key_human_clocks:
        print(f"- {clock}")
    
    # Merge metadata with clock output
    merged_df_hs = pd.merge(metadata_df_hs, clock_output_of_interest_hs, right_on="gsm_id", left_on="GSM")
    print(f"\nMerged dataset shape: {merged_df_hs.shape}")

### Human Clock Analysis and Visualization

In [ ]:
if meta_exists_hs and clock_exists_hs:
    # Identify potential comparison groups in the human dataset
    print("Identifying potential comparison groups...")
    
    # Look for columns that might indicate treatment groups
    potential_group_cols = []
    for col in merged_df_hs.columns:
        if col not in ['GSM', 'gsm_id'] and merged_df_hs[col].dtype == 'object':
            unique_vals = merged_df_hs[col].unique()
            if 2 <= len(unique_vals) <= 5:  # Good for group comparisons
                potential_group_cols.append(col)
                print(f"\n{col}: {unique_vals}")
    
    # Select the most suitable grouping variable
    if potential_group_cols:
        # Prefer columns with treatment-like names
        treatment_cols = [col for col in potential_group_cols 
                         if any(keyword in col.lower() for keyword in ['treatment', 'group', 'condition', 'status'])]
        
        if treatment_cols:
            group_col = treatment_cols[0]
        else:
            group_col = potential_group_cols[0]
            
        print(f"\nSelected grouping variable: {group_col}")
        print(f"Groups: {merged_df_hs[group_col].unique()}")
        
        # Check sample sizes
        group_counts = merged_df_hs[group_col].value_counts()
        print(f"\nSample sizes per group:")
        print(group_counts)
        
    else:
        print("No suitable grouping variables found for comparison")

In [ ]:
# Create a flexible function to analyze clock differences between any two groups
def analyze_clock_differences_flexible(data, clock_name, group_col, group1, group2):
    """
    Analyze differences between two specific groups in a dataset
    """
    # Split data into the two specified groups
    group1_data = data[data[group_col] == group1][clock_name].dropna()
    group2_data = data[data[group_col] == group2][clock_name].dropna()
    
    if len(group1_data) == 0 or len(group2_data) == 0:
        return None
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(group1_data, group2_data)
    
    # Calculate effect size (Cohen's d)
    n1, n2 = len(group1_data), len(group2_data)
    pooled_std = np.sqrt(((n1 - 1) * group1_data.std() ** 2 + (n2 - 1) * group2_data.std() ** 2) / (n1 + n2 - 2))
    
    if pooled_std == 0:
        cohens_d = 0
    else:
        cohens_d = (group2_data.mean() - group1_data.mean()) / pooled_std

    return {
        "group1_name": group1,
        "group2_name": group2,
        "group1_mean": group1_data.mean(), 
        "group2_mean": group2_data.mean(), 
        "difference": group2_data.mean() - group1_data.mean(), 
        "t_statistic": t_stat,
        "p_value": p_value,
        "cohens_d": cohens_d,
        "group1_n": n1,
        "group2_n": n2
    }

### Execute Human Data Analysis

In [ ]:
if meta_exists_hs and clock_exists_hs and 'group_col' in locals():
    # Perform analysis for each selected human clock
    print("=== HUMAN RNA CLOCK ANALYSIS RESULTS ===\n")
    
    # Get the two groups to compare (use first two groups if more than 2)
    groups = merged_df_hs[group_col].unique()
    if len(groups) >= 2:
        group1, group2 = groups[0], groups[1]
        
        print(f"Comparing: {group1} vs {group2}")
        print(f"Using grouping variable: {group_col}\n")
        
        # Create results directory
        os.makedirs("../results", exist_ok=True)
        
        # Analyze each clock
        for i, clock in enumerate(key_human_clocks):
            print(f"\n{'='*50}")
            print(f"Analyzing Human Clock {i+1}: {clock}")
            print('='*50)
            
            # Perform statistical analysis
            results = analyze_clock_differences_flexible(merged_df_hs, clock, group_col, group1, group2)
            
            if results is None:
                print("Insufficient data for analysis")
                continue
                
            # Print results
            print(f"Group 1 ({results['group1_name']}) mean: {results['group1_mean']:.3f} (n={results['group1_n']})")
            print(f"Group 2 ({results['group2_name']}) mean: {results['group2_mean']:.3f} (n={results['group2_n']})")
            print(f"Difference (Group2 - Group1): {results['difference']:.3f}")
            print(f"T-statistic: {results['t_statistic']:.3f}")
            print(f"P-value: {results['p_value']:.3f}")
            print(f"Effect size (Cohen's d): {results['cohens_d']:.3f}")
            
            # Interpret effect size
            abs_d = abs(results['cohens_d'])
            if abs_d < 0.2:
                effect_size_interp = "negligible"
            elif abs_d < 0.5:
                effect_size_interp = "small"
            elif abs_d < 0.8:
                effect_size_interp = "medium"
            else:
                effect_size_interp = "large"
            print(f"Effect size interpretation: {effect_size_interp}")
            
            # Create visualization
            plt.figure(figsize=(10, 6))
            sns.violinplot(data=merged_df_hs, x=group_col, y=clock)
            plt.title(f"Distribution of {clock}\nby {group_col}")
            plt.xlabel(group_col)
            plt.ylabel("Clock Value")
            plt.xticks(rotation=45)
            
            # Add individual points
            sns.stripplot(data=merged_df_hs, x=group_col, y=clock, color="black", alpha=0.4, size=5)
            
            plt.tight_layout()
            plt.show()
            
    else:
        print("Not enough groups for comparison")
        
else:
    print("Cannot perform human analysis - missing data or grouping variables")

### Create Human Analysis Summary Report

In [ ]:
if meta_exists_hs and clock_exists_hs and 'group_col' in locals() and len(groups) >= 2:
    # Create a summary DataFrame of human analysis results
    summary_results_hs = []
    
    for clock in key_human_clocks:
        results = analyze_clock_differences_flexible(merged_df_hs, clock, group_col, group1, group2)
        
        if results is not None:
            # Create standardized statistical output for human data
            statistical_record = {
                # Metadata
                "dataset_id": id_of_interest_hs,
                "clock_name": clock,
                "species": "Human",
                "comparison_type": "two_group",
                "group1_name": results["group1_name"],
                "group2_name": results["group2_name"],
                "grouping_variable": group_col,
                # Statistical information
                "statistical_test": "independent_t_test",
                "effect_size_type": "cohens_d",
                "effect_size": results["cohens_d"],
                "test_statistic_type": "t_statistic",
                "test_statistic": results["t_statistic"],
                "p_value": results["p_value"],
                # Group statistics
                "group1_n": results["group1_n"],
                "group2_n": results["group2_n"],
                "group1_mean": results["group1_mean"],
                "group2_mean": results["group2_mean"],
                "mean_difference": results["difference"],
                # Additional context
                "analysis_date": pd.Timestamp.now().strftime("%Y-%m-%d"),
                "analysis_type": "Human_RNA_Clock_Analysis"
            }
            summary_results_hs.append(statistical_record)

    # Create the summary DataFrame
    if summary_results_hs:
        summary_df_hs = pd.DataFrame(summary_results_hs)
        
        # Save results
        # summary_df_hs.to_csv("../results/human_statistical_summary.csv", index=False)
        
        print("\n" + "="*60)
        print("HUMAN RNA CLOCK ANALYSIS SUMMARY")
        print("="*60)
        print(f"Dataset: {id_of_interest_hs}")
        print(f"Comparison: {group1} vs {group2}")
        print(f"Number of clocks analyzed: {len(summary_df_hs)}")
        print(f"Total samples: {summary_df_hs['group1_n'].iloc[0] + summary_df_hs['group2_n'].iloc[0]}")
        
        print("\nKey Findings:")
        for _, row in summary_df_hs.iterrows():
            significance = "significant" if row['p_value'] < 0.05 else "not significant"
            direction = "higher" if row['mean_difference'] > 0 else "lower"
            print(f"- {row['clock_name']}: {row['group2_name']} shows {direction} values ({significance}, p={row['p_value']:.3f})")
        
        # print(f"\nResults saved to: ../results/human_statistical_summary.csv")
        
        # Display summary table
        print("\nSummary Table:")
        display_cols = ["clock_name", "group1_mean", "group2_mean", "mean_difference", "p_value", "effect_size"]
        print(summary_df_hs[display_cols].round(3))
        
    else:
        print("No valid results to summarize")
        
else:
    print("Human analysis could not be completed - check data availability")

In [ ]:
summary_df_hs.head()